# 12 数据质量检查

**用途**：针对粘贴板虫子图片的4类质量检查：
1. 文件名格式合规
2. 图片元数据完整
3. GT 坐标边界合规
4. 空标注统计

> `ENABLE_AUTO_TAG=False` 安全默认。


In [ ]:
# ── 0. 配置区 ──────────────────────────────────────────────
DATASET_NAME     = "sahi_null_v2_ms1_0605-0621_40_ok"
GT_FIELD         = "ground_truth"
FILENAME_YEAR    = 2024
MIN_IMAGE_SIZE   = 3400           # 宽或高小于此值视为异常
TAG_BAD_FILENAME = "bad_filename"
TAG_BAD_BBOX     = "bad_bbox"
ENABLE_AUTO_TAG  = False         # 改为 True 才自动打 Tag
# OUTPUT_PATH      = f"."


In [ ]:
# ── 1. 初始化日志 ──────────────────────────────────────────
import logging
try:
    import ipynbname
    _nb_name = ipynbname.name()
except Exception:
    _nb_name = "12_data_quality_check"

# 检查 OUTPUT_PATH 是否定义，如果没有定义则使用默认路径
try:    OUTPUT_PATH
except NameError:
    OUTPUT_PATH = f"."

log_file_name = f"{OUTPUT_PATH}/{_nb_name}_{DATASET_NAME}.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(log_file_name, encoding="utf-8"),
        logging.StreamHandler(),
    ],
)
logger = logging.getLogger(__name__)
logger.info("============ 12_data_quality_check 初始化完成 ============")


2026-02-20 16:14:23,935 [INFO] ============ 12_data_quality_check 初始化完成 ============


## Step 1: 加载数据集，计算元数据


In [22]:
import fiftyone as fo
import pandas as pd
import re
from pathlib import Path
from fiftyone import ViewField as F
from IPython.display import display

logger.info("Step 1 开始：加载数据集并计算元数据")

ds = fo.load_dataset(DATASET_NAME)
print(f"数据集: {ds.name}，样本数: {len(ds)}")
ds.compute_metadata()

all_filepaths = ds.values("filepath")
all_ids       = ds.values("id")
logger.info(f"Step 1 完成：{len(ds)} 个样本")


2026-02-20 16:14:23,941 [INFO] Step 1 开始：加载数据集并计算元数据
2026-02-20 16:14:23,949 [INFO] Step 1 完成：243 个样本


数据集: sahi_null_v2_ms1_0605-0621_40_ok，样本数: 243


## Step 2: 检查文件名格式


In [23]:
# 命名格式：MMDD_HHMM_FOCUS.jpg / .png
_FILENAME_RE = re.compile(r"(\d{4})_(\d{4})_(\d+)\.(jpg|png)$", re.IGNORECASE)

logger.info("Step 2 开始：检查文件名格式")

bad_filename_rows = []
for fid, fp in zip(all_ids, all_filepaths):
    filename = Path(fp).name
    if not _FILENAME_RE.search(filename):
        bad_filename_rows.append({"id": fid, "filepath": fp, "filename": filename})

bad_filename_df = pd.DataFrame(bad_filename_rows)
print(f"文件名格式不合规: {len(bad_filename_df)} / {len(ds)} 个")
if len(bad_filename_df) > 0:
    display(bad_filename_df.head(10))

logger.info(f"Step 2 完成：{len(bad_filename_df)} 个文件名不合规")


2026-02-20 16:14:23,953 [INFO] Step 2 开始：检查文件名格式
2026-02-20 16:14:23,955 [INFO] Step 2 完成：0 个文件名不合规


文件名格式不合规: 0 / 243 个


## Step 3: 检查图片尺寸异常


In [24]:
logger.info(f"Step 3 开始：检查图片尺寸（MIN={MIN_IMAGE_SIZE}）")

small_view = ds.filter_field("metadata.width", F() < MIN_IMAGE_SIZE)
small_h_view = ds.filter_field("metadata.height", F() < MIN_IMAGE_SIZE)

all_widths  = ds.values("metadata.width")
all_heights = ds.values("metadata.height")

small_rows = [
    {"id": fid, "filepath": fp, "width": w, "height": h}
    for fid, fp, w, h in zip(all_ids, all_filepaths, all_widths, all_heights)
    if (w is not None and w < MIN_IMAGE_SIZE) or (h is not None and h < MIN_IMAGE_SIZE)
]
small_df = pd.DataFrame(small_rows)
print(f"尺寸异常（<{MIN_IMAGE_SIZE}px）: {len(small_df)} 个")
if len(small_df) > 0:
    display(small_df)

logger.info(f"Step 3 完成：{len(small_df)} 个尺寸异常")


2026-02-20 16:14:23,961 [INFO] Step 3 开始：检查图片尺寸（MIN=3400）
2026-02-20 16:14:23,965 [INFO] Step 3 完成：0 个尺寸异常


尺寸异常（<3400px）: 0 个


## Step 4: 检查 GT 坐标合规性


In [25]:
logger.info("Step 4 开始：检查 GT bbox 坐标范围")

bad_bbox_rows = []

if GT_FIELD in ds.get_field_schema():
    for sample in ds.iter_samples():
        dets_obj = getattr(sample, GT_FIELD, None)
        if dets_obj is None or not dets_obj.detections:
            continue
        for det in dets_obj.detections:
            x, y, w, h = det.bounding_box
            is_bad = not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1
                          and x + w <= 1.001 and y + h <= 1.001)
            if is_bad:
                bad_bbox_rows.append({
                    "filepath": sample.filepath,
                    "label":    det.label,
                    "x": round(x, 4), "y": round(y, 4),
                    "w": round(w, 4), "h": round(h, 4),
                })

    bad_bbox_df = pd.DataFrame(bad_bbox_rows)
    print(f"GT bbox 坐标越界: {len(bad_bbox_df)} 个检测框")
    if len(bad_bbox_df) > 0:
        display(bad_bbox_df.head(10))
else:
    bad_bbox_df = pd.DataFrame()
    print(f"GT_FIELD='{GT_FIELD}' 不存在，跳过 bbox 检查")
    logger.warning(f"Step 4 跳过：GT_FIELD='{GT_FIELD}' 不存在")

logger.info(f"Step 4 完成：{len(bad_bbox_df)} 个 bbox 越界")


2026-02-20 16:14:23,970 [INFO] Step 4 开始：检查 GT bbox 坐标范围
2026-02-20 16:14:25,723 [INFO] Step 4 完成：0 个 bbox 越界


GT bbox 坐标越界: 0 个检测框


## Step 5: 统计空标注样本


In [26]:
logger.info("Step 5 开始：统计空标注样本")

if GT_FIELD in ds.get_field_schema():
    all_dets = ds.values(f"{GT_FIELD}.detections")
    empty_ids = [
        fid for fid, d in zip(all_ids, all_dets)
        if d is None or len(d) == 0
    ]
    print(f"空标注样本: {len(empty_ids)} / {len(ds)} 个")
    logger.info(f"Step 5 完成：空标注 {len(empty_ids)} 个")
else:
    empty_ids = []
    print(f"GT_FIELD='{GT_FIELD}' 不存在，跳过")
    logger.warning(f"Step 5 跳过：GT_FIELD 不存在")


2026-02-20 16:14:25,728 [INFO] Step 5 开始：统计空标注样本
2026-02-20 16:14:25,821 [INFO] Step 5 完成：空标注 0 个


空标注样本: 0 / 243 个


## Step 6: 汇总问题报告


In [27]:
logger.info("Step 6 开始：汇总问题报告")

bad_filename_ids = set(bad_filename_df["id"].tolist()) if not bad_filename_df.empty else set()
bad_bbox_files   = set(bad_bbox_df["filepath"].tolist()) if not bad_bbox_df.empty else set()
empty_id_set     = set(empty_ids)

summary_rows = []
for fid, fp in zip(all_ids, all_filepaths):
    summary_rows.append({
        "id":              fid,
        "filepath":        fp,
        "bad_filename":    fid in bad_filename_ids,
        "bad_bbox":        fp in bad_bbox_files,
        "empty_gt":        fid in empty_id_set,
    })

summary_df = pd.DataFrame(summary_rows)
problem_df = summary_df[summary_df[["bad_filename","bad_bbox","empty_gt"]].any(axis=1)]

print(f"有问题的样本: {len(problem_df)} / {len(summary_df)} 个")
display(problem_df)

logger.info(f"Step 6 完成：problem_df shape={problem_df.shape}")


2026-02-20 16:14:25,826 [INFO] Step 6 开始：汇总问题报告


有问题的样本: 0 / 243 个


,id,filepath,bad_filename,bad_bbox,empty_gt


2026-02-20 16:14:25,829 [INFO] Step 6 完成：problem_df shape=(0, 5)


## Step 7: 自动打 Tag（ENABLE_AUTO_TAG=True 才执行）


In [28]:
logger.info(f"Step 7 开始：ENABLE_AUTO_TAG={ENABLE_AUTO_TAG}")

if ENABLE_AUTO_TAG:
    if bad_filename_ids:
        bad_fn_view = ds.select(list(bad_filename_ids))
        bad_fn_view.tag_samples(TAG_BAD_FILENAME)
        logger.info(f"已给 {len(bad_filename_ids)} 个样本打 tag='{TAG_BAD_FILENAME}'")

    if bad_bbox_files:
        bad_bbox_sample_ids = [fid for fid, fp in zip(all_ids, all_filepaths) if fp in bad_bbox_files]
        bad_bbox_view = ds.select(bad_bbox_sample_ids)
        bad_bbox_view.tag_samples(TAG_BAD_BBOX)
        logger.info(f"已给 {len(bad_bbox_sample_ids)} 个样本打 tag='{TAG_BAD_BBOX}'")

    print("Tag 已添加，打开 App 查看")
    session = fo.launch_app(ds)
    logger.info("Step 7 完成")
else:
    print("ENABLE_AUTO_TAG=False，跳过打 Tag 操作")
    logger.info("Step 7 跳过：ENABLE_AUTO_TAG=False")


2026-02-20 16:14:25,836 [INFO] Step 7 开始：ENABLE_AUTO_TAG=False
2026-02-20 16:14:25,836 [INFO] Step 7 跳过：ENABLE_AUTO_TAG=False


ENABLE_AUTO_TAG=False，跳过打 Tag 操作
